# Single Product Tournament Harness

End-to-end single-product walkthrough for the **tournament-first** Product Evidence Harness.

Primary flow:

```text
Input product identity
  → SerpAPI search fan-out, capped at 4 credits
  → candidate pool
  → batch scraping
  → batch winners
  → production-ready champion candidate
  → champion confirmation gate, default 3 checks
  → production URL gate
  → product-coding evidence bundle
```

Production handoff rule:

```text
production_url_ready == True
production_url_status == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
champion_confirmation.passed == True
champion_confirmation.success_count == 3
needs_review == False
```

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

In [ ]:
from product_evidence_harness import (
    HarnessConfig,
    ProductEvidenceHarness,
    ProductQuery,
    ProductionURLGate,
    SerpAPIConfig,
    configure_logging,
)

configure_logging("INFO")

In [ ]:
product = ProductQuery(
    row_id="CO-ML-0001",
    main_text="PUT PRODUCT TEXT HERE",
    country_code="CO",
    ean="",  # optional; keep EAN/GTIN as text
    retailer_name="Mercado Libre",  # optional preferred-first source
)
product

In [ ]:
config = HarnessConfig.from_env(PROJECT_ROOT / ".env")
assert config.tournament.enabled is True, "Tournament mode should be enabled for the primary harness flow."
assert config.tournament.max_serp_credits <= 4, "Tournament mode must respect the 4-credit SerpAPI cap."

pd.Series({
    "tournament_enabled": config.tournament.enabled,
    "max_serp_credits": config.tournament.max_serp_credits,
    "candidate_pool": config.tournament.candidate_pool,
    "preflight_top_k": config.tournament.preflight_top_k,
    "batch_size": config.tournament.batch_size,
    "max_batches": config.tournament.max_batches,
    "early_stop": config.tournament.early_stop,
    "champion_confirmation_attempts": 3,
    "champion_confirmation_required_successes": 3,
}).to_frame("value")

In [ ]:
serp_config = SerpAPIConfig.from_env(
    country_code=product.country_code,
    language_code=product.language_code or "es",
    env_file=PROJECT_ROOT / ".env",
)

harness = ProductEvidenceHarness(serp_config=serp_config, config=config)
trace = harness.run(product, return_trace=True)
match = trace.best_match

In [ ]:
tournament = getattr(trace.state, "tournament_result", None)
confirmation = getattr(tournament, "champion_confirmation", None) if tournament else None
production = ProductionURLGate().assess_url_in_state(trace.state, match.product_url or "")

summary = {
    "product_url": match.product_url,
    "verified_exact_url": match.verified_exact_url,
    "production_url_ready": production.production_ready if production else False,
    "production_url_status": production.status if production else "PRODUCT_URL_NOT_ASSESSED_OR_NO_SCORECARD",
    "browser_openable": production.browser_openable if production else False,
    "highly_scrapable": production.highly_scrapable if production else False,
    "exact_product_url_match": production.exact_product_match if production else False,
    "production_url_score": production.score if production else 0.0,
    "production_url_reasons": " | ".join(production.reasons) if production else "NO_SCORECARD_FOR_SELECTED_PRODUCT_URL",
    "champion_confirmation_passed": confirmation.passed if confirmation else False,
    "champion_confirmation_status": confirmation.status if confirmation else "NO_CONFIRMATION",
    "champion_confirmation_attempted_count": confirmation.attempted_count if confirmation else 0,
    "champion_confirmation_required_attempts": confirmation.required_attempts if confirmation else 3,
    "champion_confirmation_success_count": confirmation.success_count if confirmation else 0,
    "champion_confirmation_required_successes": confirmation.required_successes if confirmation else 3,
    "champion_final_url_stable": confirmation.final_url_stable if confirmation else False,
    "champion_evidence_stable": confirmation.evidence_stable if confirmation else False,
    "champion_min_richness": confirmation.min_richness if confirmation else 0.0,
    "champion_min_word_count": confirmation.min_word_count if confirmation else 0,
    "needs_review": match.needs_review,
    "url_decision_status": match.url_decision_status,
    "confidence": match.confidence,
    "tournament_champion_url": tournament.champion_url if tournament else None,
    "tournament_runner_up_url": tournament.runner_up_url if tournament else None,
    "tournament_champion_margin": tournament.champion_margin if tournament else None,
    "tournament_search_credits_used": tournament.search_credits_used if tournament else None,
    "tournament_search_credit_limit": tournament.search_credit_limit if tournament else None,
}
pd.Series(summary).to_frame("value")

In [ ]:
handoff_ready = bool(
    production
    and production.production_ready
    and production.status == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
    and confirmation
    and confirmation.passed
    and confirmation.success_count >= confirmation.required_successes
    and not match.needs_review
)

print("HANDOFF_READY_FOR_BROWSER_AND_SCRAPING_TEAM =", handoff_ready)
if not handoff_ready:
    print("This product_url is available but review-only / not production handoff-ready.")

In [ ]:
row_dir = Path(config.output_dir) / product.row_id
print("Row artifact directory:", row_dir)
expected = [
    "final_row.csv",
    "tournament_bracket.md",
    "tournament_bracket.json",
    "champion_confirmation.md",
    "champion_confirmation.json",
    "batch_winners.csv",
    "quality_assessment.md",
    "product_coding_input.json",
    "evidence_graph.json",
]
if row_dir.exists():
    existing = {p.name for p in row_dir.iterdir()}
    display(pd.DataFrame({"artifact": expected, "exists": [name in existing for name in expected]}))
else:
    print("Row directory not found. Check PRODUCT_HARNESS_OUTPUT_DIR.")

In [ ]:
confirmation_path = row_dir / "champion_confirmation.json"
if confirmation_path.exists():
    confirmation_payload = json.loads(confirmation_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "url": confirmation_payload.get("url"),
        "passed": confirmation_payload.get("passed"),
        "status": confirmation_payload.get("status"),
        "attempted_count": confirmation_payload.get("attempted_count"),
        "required_attempts": confirmation_payload.get("required_attempts"),
        "success_count": confirmation_payload.get("success_count"),
        "required_successes": confirmation_payload.get("required_successes"),
        "final_url_stable": confirmation_payload.get("final_url_stable"),
        "evidence_stable": confirmation_payload.get("evidence_stable"),
        "min_richness": confirmation_payload.get("min_richness"),
        "min_word_count": confirmation_payload.get("min_word_count"),
    }, indent=2, ensure_ascii=False))
    display(pd.DataFrame(confirmation_payload.get("attempts", [])))
else:
    print("champion_confirmation.json not found")